# Week 11 extra: LoRA rank ablation (3-class)
**TMI-102 | PaAC, IIT Roorkee** — runs on Kaggle (GPU on)

**Question:** how does LoRA's test accuracy change with the rank r (1, 2, 4, 8, 16)?

**Same setup as the retrain run** (Week 6 LoRA): same 3-class data and split, same augmentation, same class weights, LR 1e-3, up to 12 epochs, early stopping (patience 4, best weights restored). Only the rank changes. `alpha` stays 32, so the scaling `alpha/r` changes with the rank, as in the original LoRA setup.

**Setup:** add the Galaxy Zoo competition data as input, turn the GPU on, run top to bottom. Results are written to `ablation_results.json` after every rank, so nothing is lost if the session stops partway. Expect roughly 1 to 2 hours in total.


In [ ]:
# ── Build Galaxy_dataset_3class from the competition files (runs once; skipped if the folder already exists) ──
import os, zipfile
import numpy as np, pandas as pd

COMP = "/kaggle/input/competitions/galaxy-zoo-the-galaxy-challenge"
BUILD_DIR = "/kaggle/working/Galaxy_dataset_3class"
ELL_T, SPI_T, MERGER_T, SEED = 0.80, 0.80, 0.30, 42     # merger = Class8.6 >= 0.30 (change here if mentors say otherwise)

if not os.path.isdir(BUILD_DIR + "/train"):
    from sklearn.model_selection import train_test_split
    df = pd.read_csv(f"{COMP}/training_solutions_rev1.zip", compression="zip")

    # --- elliptical + spiral: EXACTLY the Week 3 recipe (same thresholds, same seed, same split) ---
    df["class"] = np.select([df["Class1.1"] >= ELL_T, df["Class1.2"] >= SPI_T], ["elliptical", "spiral"], default="Tie")
    filtered = df[df["class"].isin(["elliptical", "spiral"])].copy()
    min_cnt = min(filtered["class"].value_counts())
    balanced = pd.concat([
        filtered[filtered["class"] == "spiral"].sample(n=min_cnt, random_state=SEED),
        filtered[filtered["class"] == "elliptical"].sample(n=min_cnt, random_state=SEED)
    ]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    tr, tmp = train_test_split(balanced, test_size=0.30, random_state=SEED, stratify=balanced["class"])
    va, te = train_test_split(tmp, test_size=0.50, random_state=SEED, stratify=tmp["class"])

    # --- merger: simply ADDED as a third class. Galaxies already used as elliptical/spiral are skipped,
    #     so one image never has two labels and the two old classes stay exactly as they were. ---
    mer_all = df[df["Class8.6"] >= MERGER_T]
    mer = mer_all[~mer_all["GalaxyID"].isin(balanced["GalaxyID"])].copy()
    mer["class"] = "merger"
    print(f"merger candidates {len(mer_all)} -> {len(mer)} after skipping {len(mer_all) - len(mer)} already used as elliptical/spiral")
    mtr, mtmp = train_test_split(mer, test_size=0.30, random_state=SEED)
    mva, mte = train_test_split(mtmp, test_size=0.50, random_state=SEED)

    splits = [(tr, "train"), (va, "val"), (te, "test"), (mtr, "train"), (mva, "val"), (mte, "test")]
    for s in ["train", "val", "test"]:
        for c in ["elliptical", "merger", "spiral"]:
            os.makedirs(f"{BUILD_DIR}/{s}/{c}", exist_ok=True)
    target, rows = {}, []
    for d, s in splits:
        for gid, c in zip(d["GalaxyID"], d["class"]):
            target[f"{gid}.jpg"] = (s, c); rows.append((gid, c, s))

    n_ext = 0
    with zipfile.ZipFile(f"{COMP}/images_training_rev1.zip") as z:
        for member in z.namelist():
            base = os.path.basename(member)
            if base in target:
                s, c = target[base]
                with open(f"{BUILD_DIR}/{s}/{c}/{base}", "wb") as f:
                    f.write(z.read(member))
                n_ext += 1
    man = pd.DataFrame(rows, columns=["GalaxyID", "cls", "split"])
    man.to_csv(f"{BUILD_DIR}/manifest.csv", index=False)
    print("extracted", n_ext, "of", len(man))
    print(man.groupby(["split", "cls"]).size().unstack())
    print("Check vs Week 3: elliptical/spiral should be 5692 / 1220 / 1220 each.")
else:
    print("Dataset already built at", BUILD_DIR)

In [ ]:
import os, glob, json, time
import numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print("TF", tf.__version__, "| GPU:", len(tf.config.list_physical_devices('GPU')) > 0)

DATASET_DIR = BUILD_DIR
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
print("Dataset:", DATASET_DIR)

In [ ]:
IMG_SIZE, BATCH_SIZE, NUM_CLASSES = 224, 32, 3

train_aug = ImageDataGenerator(preprocessing_function=preprocess_input,
                               rotation_range=360, horizontal_flip=True, vertical_flip=True)
plain = ImageDataGenerator(preprocessing_function=preprocess_input)

def make(gen, split, shuffle):
    return gen.flow_from_directory(f'{DATASET_DIR}/{split}', target_size=(IMG_SIZE, IMG_SIZE),
                                   batch_size=BATCH_SIZE, class_mode='categorical', shuffle=shuffle)

train_gen = make(train_aug, 'train', True)
val_gen   = make(plain, 'val', False)
test_gen  = make(plain, 'test', False)

print(train_gen.class_indices)          # expect {'elliptical': 0, 'merger': 1, 'spiral': 2}
CLASS_NAMES = [c for c, _ in sorted(train_gen.class_indices.items(), key=lambda kv: kv[1])]

# class weight = N / (K * count_of_that_class)  -> rare class counts more in the loss
counts = np.bincount(train_gen.classes)
class_weight = {i: len(train_gen.classes) / (NUM_CLASSES * n) for i, n in enumerate(counts)}
for i, c in enumerate(CLASS_NAMES):
    print(f"{c:>10}: {counts[i]:5d} train images -> weight {class_weight[i]:.3f}")

In [ ]:
@tf.keras.utils.register_keras_serializable(package='tmi102')
class LoRADense(layers.Layer):
    """Dense layer with a LoRA adapter: output = x@W + b + (x@A@B) * (alpha/rank).  W, b frozen; A, B trainable."""
    def __init__(self, units, rank=4, alpha=32, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units, self.rank, self.alpha = units, rank, alpha
        self.scaling = alpha / rank
        self.activation_fn = tf.keras.activations.get(activation)

    def build(self, input_shape):
        input_dim = input_shape[-1]
        self.W = self.add_weight(name='kernel', shape=(input_dim, self.units), initializer='glorot_uniform', trainable=False)
        self.b = self.add_weight(name='bias', shape=(self.units,), initializer='zeros', trainable=False)
        self.A = self.add_weight(name='lora_A', shape=(input_dim, self.rank), initializer='glorot_normal', trainable=True)
        self.B = self.add_weight(name='lora_B', shape=(self.rank, self.units), initializer='zeros', trainable=True)

    def call(self, x):
        h = tf.matmul(x, self.W) + self.b
        h = h + tf.matmul(tf.matmul(x, self.A), self.B) * self.scaling
        return self.activation_fn(h) if self.activation_fn else h

    def get_config(self):
        cfg = super().get_config()
        cfg.update(units=self.units, rank=self.rank, alpha=self.alpha,
                   activation=tf.keras.activations.serialize(self.activation_fn))
        return cfg

## Rank ablation: one LoRA model per rank

In [ ]:
import random
RANKS = [1, 2, 4, 8, 16]
ABL_PATH = f'{OUT_DIR}/ablation_results.json'

def build_lora(rank):
    base = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = LoRADense(256, rank=rank, alpha=32, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return Model(inputs, outputs, name=f'resnet50_lora_r{rank}')

def run_rank(rank, lr=1e-3, epochs=12, patience=4):
    tf.keras.backend.clear_session()
    random.seed(42); np.random.seed(42); tf.random.set_seed(42)
    model = build_lora(rank)
    n_train = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    print(f"\n===== rank {rank} | trainable {n_train:,} =====")
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='categorical_crossentropy', metrics=['accuracy'])
    cb = [EarlyStopping(patience=patience, restore_best_weights=True)]
    t0 = time.time()
    hist = model.fit(train_gen, validation_data=val_gen, epochs=epochs, class_weight=class_weight, callbacks=cb)
    n_ep = len(hist.history['loss'])
    best_ep = int(np.argmin(hist.history['val_loss'])) + 1
    test_loss, test_acc = model.evaluate(test_gen, verbose=0)
    test_gen.reset()
    p = model.predict(test_gen, verbose=0).argmax(1)
    y = test_gen.classes
    cm = confusion_matrix(y, p)
    rec = cm.diagonal() / cm.sum(1)
    res = json.load(open(ABL_PATH)) if os.path.exists(ABL_PATH) else {}
    res[str(rank)] = dict(rank=rank, trainable_params=n_train, test_acc=float(test_acc), test_loss=float(test_loss),
                          macro_f1=float(f1_score(y, p, average='macro')),
                          recall={c: float(r) for c, r in zip(CLASS_NAMES, rec)},
                          epochs_run=n_ep, best_epoch=best_ep, sec_per_epoch=float((time.time() - t0) / n_ep),
                          history={k: [float(v) for v in vals] for k, vals in hist.history.items()})
    json.dump(res, open(ABL_PATH, 'w'), indent=2)
    print(f"rank {rank}: test acc {test_acc * 100:.2f}% | macro-F1 {res[str(rank)]['macro_f1']:.4f} | merger recall {rec[1] * 100:.1f}% | {n_ep} epochs (best {best_ep})")

for r in RANKS:
    done = json.load(open(ABL_PATH)) if os.path.exists(ABL_PATH) else {}
    if str(r) in done:
        print(f"rank {r} already done, skipping"); continue
    run_rank(r)


## Summary table and curve

In [ ]:
res = json.load(open(ABL_PATH))
print(f"{'rank':>5}{'params':>10}{'test acc':>10}{'macro-F1':>10}{'merger rec':>12}{'epochs':>8}{'best ep':>9}")
for r in RANKS:
    d = res[str(r)]
    print(f"{r:>5}{d['trainable_params']:>10,}{d['test_acc'] * 100:>9.2f}%{d['macro_f1']:>10.4f}{d['recall']['merger'] * 100:>11.1f}%{d['epochs_run']:>8}{d['best_epoch']:>9}")

plt.figure(figsize=(6, 4))
plt.plot(RANKS, [res[str(r)]['test_acc'] * 100 for r in RANKS], 'o-', color='#a78bfa', lw=2)
plt.axhline(93.17, color='#34d399', ls='--', label='Full FT (93.17%)')
plt.xscale('log', base=2); plt.xticks(RANKS, [str(r) for r in RANKS])
plt.xlabel('LoRA rank (r)'); plt.ylabel('Test accuracy (%)'); plt.title('LoRA rank ablation (3-class)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
